In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bars_path = "dbfs:/mnt/crypto/gold/crypto_1m_bars"

# 1) read your 1m bars
bars_df = (
    spark.read.format("delta")
         .load(bars_path)
         .orderBy("symbol", "window_start")
)

# 2) define window per symbol ordered by time
w = Window.partitionBy("symbol").orderBy("window_start")

# 3) make lag features (last 3 minutes for now)
feat_df = (
    bars_df
    .withColumn("close_lag_1", F.lag("close_price", 1).over(w))
    .withColumn("close_lag_2", F.lag("close_price", 2).over(w))
    .withColumn("close_lag_3", F.lag("close_price", 3).over(w))
    # target = next minute’s close
    .withColumn("target_next_close", F.lead("close_price", 1).over(w))
)

# 4) drop rows that don’t have full history/target
train_df = (
    feat_df
    .dropna(subset=["close_lag_1", "close_lag_2", "close_lag_3", "target_next_close"])
)

# 5) write it out so the training notebook can read it
features_path = "dbfs:/mnt/crypto/ml/features_1m"
(
    train_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(features_path)
)

print("✅ features written to", features_path)

✅ features written to dbfs:/mnt/crypto/ml/features_1m


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

features_path = "dbfs:/mnt/crypto/ml/features_1m"

# 1) read features
df = spark.read.format("delta").load(features_path)

# 2) train on just BTC for now (we'll loop over symbols later)
symbol_to_train = "BTC"
train_df = df.filter(F.col("symbol") == symbol_to_train)

# 3) assemble features
feature_cols = ["close_lag_1", "close_lag_2", "close_lag_3"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# 4) simple regression to predict next minute close
lr = LinearRegression(
    featuresCol="features",
    labelCol="target_next_close",
    predictionCol="prediction"
)

pipeline = Pipeline(stages=[assembler, lr])

# 5) split & train
train_data, test_data = train_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_data)

# 6) quick look at predictions
preds = model.transform(test_data)
preds.select("symbol", "window_start", "target_next_close", "prediction").show(20, truncate=False)

# 7) save model
model_path = "dbfs:/mnt/crypto/ml/models/next_minute_lr_btc"
model.write().overwrite().save(model_path)

print("✅ model trained and saved to", model_path)

🏃 View run honorable-mink-226 at: https://adb-3413485257045468.8.azuredatabricks.net/ml/experiments/592b264b77a54062b50e7d688b995212/runs/a6e2d75fbe6742a493f69563c7776da1
🧪 View experiment at: https://adb-3413485257045468.8.azuredatabricks.net/ml/experiments/592b264b77a54062b50e7d688b995212
+------+-------------------+-----------------+------------------+
|symbol|window_start       |target_next_close|prediction        |
+------+-------------------+-----------------+------------------+
|BTC   |2025-11-10 06:33:00|106331.0547      |106118.17752100456|
|BTC   |2025-11-10 06:36:00|106289.4771      |106385.9103023249 |
|BTC   |2025-11-10 06:38:00|106149.8826      |106336.36139730277|
|BTC   |2025-11-10 06:46:00|106407.0342      |106359.43646646164|
+------+-------------------+-----------------+------------------+

✅ model trained and saved to dbfs:/mnt/crypto/ml/models/next_minute_lr_btc


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel

# paths
features_path    = "dbfs:/mnt/crypto/ml/features_1m"
predictions_path = "dbfs:/mnt/crypto/gold/predictions_next_minute"

# symbols we trained
symbols = ["BTC", "ETH", "SOL", "LTC", "XRP"]

# 1) load all features
feat_df = spark.read.format("delta").load(features_path)

# 2) optional: take only the most recent N per symbol (faster)
N = 200  # change if you want more history in predictions
w = Window.partitionBy("symbol").orderBy(F.col("window_start").desc())

feat_recent = (
    feat_df
      .withColumn("rn", F.row_number().over(w))
      .filter(F.col("rn") <= N)
      .drop("rn")
)

all_preds = None

# 3) loop over each symbol, load its model, predict
for sym in symbols:
    model_path = f"dbfs:/mnt/crypto/ml/models/next_minute_lr_{sym.lower()}"
    model = PipelineModel.load(model_path)

    feat_sym = (
        feat_recent
          .filter(F.col("symbol") == sym)
          .orderBy(F.col("window_start").desc())
    )

    preds_sym = (
        model.transform(feat_sym)
             .select(
                 F.lit(sym).alias("symbol"),
                 "window_start",
                 F.col("prediction").alias("predicted_next_close"),
                 "target_next_close"
             )
    )

    all_preds = preds_sym if all_preds is None else all_preds.unionByName(preds_sym)

# 4) write ALL coins’ predictions to delta (overwrite old version)
(
    all_preds
      .write
      .format("delta")
      .mode("overwrite")
      .save(predictions_path)
)

print("✅ wrote multi-coin predictions to", predictions_path)

✅ wrote multi-coin predictions to dbfs:/mnt/crypto/gold/predictions_next_minute


In [0]:
# Snowflake connection options
sfOptions = {
    "sfURL": "os76797.central-us.azure.snowflakecomputing.com",
    "sfUser": "AJAYSURVE2024",
    "sfPassword": "Lionelmessi&10",     # TODO: move to secret later
    "sfDatabase": "CRYPTO_ANALYTICS",
    "sfSchema": "PUBLIC",
    "sfWarehouse": "COMPUTE_WH",             # or COMPUTE_WH if that's what you have
    "sfRole": "ACCOUNTADMIN"
}

preds_df = spark.read.format("delta").load("dbfs:/mnt/crypto/gold/predictions_next_minute")

(
    preds_df.write
        .format("net.snowflake.spark.snowflake")
        .options(**sfOptions)
        .option("dbtable", "PREDICTIONS_NEXT_MINUTE")
        .mode("overwrite")   # during dev; later you can 'append'
        .save()
)

print("✅ pushed multi-coin predictions to Snowflake")

✅ pushed multi-coin predictions to Snowflake


In [0]:
spark.read.format("delta") \
     .load("dbfs:/mnt/crypto/gold/predictions_next_minute") \
     .groupBy("symbol").count().show()

+------+-----+
|symbol|count|
+------+-----+
|   BTC|   23|
|   XRP|   23|
|   SOL|   23|
|   LTC|   23|
|   ETH|   23|
+------+-----+



In [0]:
sfOptions = {
    "sfURL": "os76797.central-us.azure.snowflakecomputing.com",  # your account + region
    "sfUser": "AJAYSURVE2024",
    "sfPassword": "Lionelmessi&10",        # or fetch from a secret scope
    "sfDatabase": "CRYPTO_ANALYTICS",
    "sfSchema": "PUBLIC",
    "sfWarehouse": "COMPUTE_WH",                # use the warehouse you created
    "sfRole": "ACCOUNTADMIN"                   # or another role you use
}

In [0]:
preds_df = spark.read.format("delta").load("dbfs:/mnt/crypto/gold/predictions_next_minute")

(
    preds_df.write
        .format("net.snowflake.spark.snowflake")
        .options(**sfOptions)
        .option("dbtable", "PREDICTIONS_NEXT_MINUTE")
        .mode("overwrite")   # overwrite so Snowflake gets all 5 coins
        .save()
)

In [0]:
sfOptions = {
  "sfURL": "os76797.central-us.azure.snowflakecomputing.com",  # from CURRENT_ACCOUNT + region
  "sfUser": "AJAYSURVE2024",
  "sfPassword": "Lionelmessi&10",
  "sfDatabase": "CRYPTO_ANALYTICS",
  "sfSchema": "PUBLIC",
  "sfWarehouse": "COMPUTE_WH",
  "sfRole": "ACCOUNTADMIN"
}

In [0]:
from pyspark.sql.functions import current_timestamp

preds_df = (
    spark.read.format("delta")
         .load("dbfs:/mnt/crypto/gold/predictions_next_minute")
         .withColumn("load_ts", current_timestamp())
)

# write to Snowflake
(
    preds_df.write
        .format("net.snowflake.spark.snowflake")
        .options(**sfOptions)
        .option("dbtable", "PREDICTIONS_NEXT_MINUTE")
        .mode("overwrite")  # first time overwrite; later switch to 'append'
        .save()
)

In [0]:
sfOptions = {
  "sfURL": "os76797.central-us.azure.snowflakecomputing.com",
  "sfUser": "AJAYSURVE2024",
  "sfPassword": "Lionelmessi&10",   # <-- put your Snowflake password here
  "sfDatabase": "CRYPTO_ANALYTICS",
  "sfSchema": "PUBLIC",
  "sfWarehouse": "COMPUTE_WH",
  "sfRole": "ACCOUNTADMIN"
}

In [0]:
from pyspark.sql.functions import current_timestamp

bars_df = (
    spark.read.format("delta")
         .load("dbfs:/mnt/crypto/gold/crypto_1m_bars")
         .withColumn("load_ts", current_timestamp())
)

In [0]:
# make sure table exists (Snowflake will create if it doesn't)
spark._jvm.net.snowflake.spark.snowflake.Utils.runQuery(
    sfOptions,
    """
    CREATE TABLE IF NOT EXISTS CRYPTO_ANALYTICS.PUBLIC.BARS_1M_ACTUAL (
        SYMBOL STRING,
        WINDOW_START TIMESTAMP,
        WINDOW_END   TIMESTAMP,
        OPEN_PRICE   DOUBLE,
        HIGH_PRICE   DOUBLE,
        LOW_PRICE    DOUBLE,
        CLOSE_PRICE  DOUBLE,
        AVG_PRICE    DOUBLE,
        LOAD_TS      TIMESTAMP
    )
    """
)

# now write
(
    bars_df.write
        .format("net.snowflake.spark.snowflake")
        .options(**sfOptions)
        .option("dbtable", "BARS_1M_ACTUAL")
        .mode("overwrite")   # first time overwrite; later use 'append'
        .save()
)

In [0]:
spark.read.format("delta").load("dbfs:/mnt/crypto/gold/predictions_next_minute").orderBy("window_start", ascending=False).show(5)

+------+-------------------+--------------------+-----------------+
|symbol|       window_start|predicted_next_close|target_next_close|
+------+-------------------+--------------------+-----------------+
|   BTC|2025-11-10 06:47:00|  106395.85539987482|       106570.441|
|   ETH|2025-11-10 06:47:00|   3620.432059174047|        3641.6608|
|   XRP|2025-11-10 06:47:00|  2.4031534230608886|           2.4077|
|   SOL|2025-11-10 06:47:00|  165.47492228554643|         165.8638|
|   LTC|2025-11-10 06:47:00|   109.1766989282687|         108.1304|
+------+-------------------+--------------------+-----------------+
only showing top 5 rows
